# IID Experiments — Single-class & Multi-class (Pavia-U)

**No PCA / No AE** — every detector receives raw 103-D bands.  
Score nets (DSM, LRao) carry a **frozen ZCA whitening first layer** (fit on training background; relative eigenvalue floor).  
Two sweeps per run:
- **vs n** at fixed ρ: AUC, partial-AUC (Pfa<0.1), Pd@Pfa
- **vs ρ** at fixed n: AUC, Pd@Pfa

Detectors: **AMF, Reg-AMF, GMM-GLRT** (classical) + **DSM, LRao** (score-based)  
Outputs: `roc_at_nmax.pdf`, `roc_at_nfixed.pdf`, `auc_vs_n.pdf`, `pauc_vs_n.pdf`, `pd_at_fa_vs_n.pdf`, `auc_vs_rho.pdf`, `pdet_at_pfa_vs_rho.pdf`

**Runtime**: single-class ≈15–30 min on T4 GPU (full config); multi-class ≈20–40 min.  
Use the *quick* overrides in Cell 3 to smoke-test in <5 min.

In [ ]:
# ── Cell 1: Clone / pull repo + install deps ──────────────────────────────
import os, sys
REPO_URL = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL    = '/content/proj'
BRANCH   = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL, '.git')):
    !git -C {LOCAL} fetch --all -q
    !git -C {LOCAL} checkout {BRANCH} -q
    !git -C {LOCAL} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml einops

for p in [LOCAL]:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(LOCAL)
print('CWD:', os.getcwd())
!git -C {LOCAL} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — running on CPU (slower). Enable GPU in Runtime > Change runtime type.')
print(f'PyTorch {torch.__version__}  device={DEVICE}')

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────────
# The dataset (pavia-u.mat) must be on your Google Drive.
# Expected location:  MyDrive/final_paper/pavia-u.mat
# (also checked at:   MyDrive/final_paper/real_datasets/pavia-u.mat)
import os

RESULTS = '/content/results'    # override below if you want Drive persistence

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/iid_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in [
            '/content/drive/MyDrive/final_paper/pavia-u.mat',
            '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat',
        ]:
            if os.path.exists(src):
                os.symlink(src, DST)
                print('Linked dataset from', src)
                break
        else:
            print('WARNING: pavia-u.mat not found on Drive — upload it or check path.')
    else:
        print('Dataset already linked:', DST)
except Exception as e:
    print('Drive not mounted:', repr(e))
    print('Using local /content/results. Dataset must be at data/pavia-u.mat.')

os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), \
    'pavia-u.mat missing! Mount Drive (above) and re-run this cell.'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Config overrides ───────────────────────────────────────────────
# Set QUICK=True for a fast smoke-test (~3 min total).
# Set QUICK=False for the full paper run (longer but complete).
QUICK = False   # ← flip to True for smoke-test

QUICK_OVERRIDES = dict(
    n_train_list    = [100, 500],
    rho_list        = [0.003, 0.01, 0.1],
    n_fixed_for_rho = 100,
    dsm_epochs      = 200,
    lrao_epochs     = 30,
    test_size       = 300,
    device          = DEVICE,
)
FULL_OVERRIDES = dict(
    device = DEVICE,
)
OVERRIDES = QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
print('Mode:', 'QUICK smoke-test' if QUICK else 'FULL paper run')
print('Overrides:', OVERRIDES)

---
## Single-class IID (bkg = asphalt cls1, tgt = trees cls3)

In [ ]:
# ── Cell S1: Run single-class IID ─────────────────────────────────────────
import yaml, os
from iid_core import run_iid

CFG_SINGLE = 'experiments/iid_single/config.yaml'
with open(CFG_SINGLE) as f:
    cfg_single = yaml.safe_load(f)

# Apply overrides (including device + optional quick settings)
cfg_single.update(OVERRIDES)
# Point results to our chosen RESULTS dir
cfg_single['results_dir'] = os.path.join(RESULTS, 'iid_single')
os.makedirs(cfg_single['results_dir'], exist_ok=True)

print('=== IID SINGLE-CLASS ===')
print(f"dataset   : {cfg_single['dataset']}")
print(f"bkg_cls   : {cfg_single['bkg_cls']}   target_cls: {cfg_single['target_cls']}")
print(f"n_list    : {cfg_single['n_train_list']}")
print(f"rho_list  : {cfg_single['rho_list']}")
print(f"dsm_epochs: {cfg_single['dsm_epochs']}")
print()

run_dir_single, metrics_single = run_iid(cfg_single, mode='single')
print('\nSingle run dir:', run_dir_single)

In [ ]:
# ── Cell S2: Display single-class figures ─────────────────────────────────
import glob
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir_s = os.path.join(run_dir_single, 'figures')

FIGS_SINGLE = [
    ('roc_at_nmax.pdf',         'ROC — largest n'),
    ('roc_at_nfixed.pdf',       'ROC — fixed n, sweep ρ'),
    ('auc_vs_n.pdf',            'AUC vs n'),
    ('pauc_vs_n.pdf',           'Partial-AUC (Pfa<0.1) vs n'),
    ('pd_at_fa_vs_n.pdf',       'Pd @ Pfa=0.1 vs n'),
    ('auc_vs_rho.pdf',          'AUC vs ρ'),
    ('pdet_at_pfa_vs_rho.pdf',  'Pd @ Pfa vs ρ'),
]

for fname, title in FIGS_SINGLE:
    pdf_path = os.path.join(fig_dir_s, fname)
    png_path = pdf_path.replace('.pdf', '.png')
    if os.path.exists(pdf_path):
        # Convert PDF → PNG for inline display
        os.system(f'convert -density 150 "{pdf_path}" "{png_path}" 2>/dev/null || '
                  f'pdftoppm -r 150 -png "{pdf_path}" "{png_path[:-4]}" 2>/dev/null')
        img_file = png_path if os.path.exists(png_path) else (
            png_path[:-4] + '-1.png' if os.path.exists(png_path[:-4] + '-1.png') else None)
        if img_file and os.path.exists(img_file):
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.imshow(mpimg.imread(img_file))
            ax.axis('off'); ax.set_title(f'[single] {title}', fontsize=10)
            plt.tight_layout(); plt.show()
        else:
            print(f'[single] {title}: PDF at {pdf_path} (convert not available for inline display)')
    else:
        print(f'[single] {title}: not found at {pdf_path}')

# Loss curve (PNG, inline directly)
loss_png = glob.glob(os.path.join(fig_dir_s, 'loss_curves_*.png'))
if loss_png:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.imshow(mpimg.imread(loss_png[0]))
    ax.axis('off'); ax.set_title('[single] Training loss curves', fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Cell S3: Single-class numeric summary ─────────────────────────────────
import json
import pandas as pd

with open(os.path.join(run_dir_single, 'metrics.json')) as f:
    m = json.load(f)

n_list_s  = m['n_list']
rho_list_s = m['rho_list']
dets = list(m['vs_n'].keys())

print('=== vs-n  AUC (single-class) ===')
df_auc_n = pd.DataFrame({d: m['vs_n'][d]['auc'] for d in dets}, index=n_list_s)
df_auc_n.index.name = 'n'
print(df_auc_n.round(3).to_string())

print('\n=== vs-ρ  AUC (single-class) ===')
df_auc_r = pd.DataFrame({d: m['vs_rho'][d]['auc'] for d in dets}, index=rho_list_s)
df_auc_r.index.name = 'rho'
print(df_auc_r.round(3).to_string())

if 'roc_at_nmax' in m:
    n_roc = n_list_s[-1]
    print(f'\n=== ROC AUC at n={n_roc} ===')
    for d, v in m['roc_at_nmax'].items():
        print(f'  {d:12s}: AUC = {v["auc"]:.4f}')

---
## Multi-class IID (tgt = trees cls3, bkg = all other classes)

In [ ]:
# ── Cell M1: Run multi-class IID ──────────────────────────────────────────
import yaml, os
from iid_core import run_iid

CFG_MULTI = 'experiments/iid_multi/config.yaml'
with open(CFG_MULTI) as f:
    cfg_multi = yaml.safe_load(f)

cfg_multi.update(OVERRIDES)
cfg_multi['results_dir'] = os.path.join(RESULTS, 'iid_multi')
os.makedirs(cfg_multi['results_dir'], exist_ok=True)

print('=== IID MULTI-CLASS ===')
print(f"target_cls : {cfg_multi['target_cls']}  (bkg = all other classes)")
print(f"n_list     : {cfg_multi['n_train_list']}")
print(f"rho_list   : {cfg_multi['rho_list']}")
print(f"gmm_K      : {cfg_multi['gmm_K']}  (larger K for mixed background)")
print()

run_dir_multi, metrics_multi = run_iid(cfg_multi, mode='multi')
print('\nMulti run dir:', run_dir_multi)

In [ ]:
# ── Cell M2: Display multi-class figures ──────────────────────────────────
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir_m = os.path.join(run_dir_multi, 'figures')

FIGS_MULTI = [
    ('roc_at_nmax.pdf',         'ROC — largest n'),
    ('roc_at_nfixed.pdf',       'ROC — fixed n, sweep ρ'),
    ('auc_vs_n.pdf',            'AUC vs n'),
    ('pauc_vs_n.pdf',           'Partial-AUC (Pfa<0.1) vs n'),
    ('pd_at_fa_vs_n.pdf',       'Pd @ Pfa=0.1 vs n'),
    ('auc_vs_rho.pdf',          'AUC vs ρ'),
    ('pdet_at_pfa_vs_rho.pdf',  'Pd @ Pfa vs ρ'),
]

for fname, title in FIGS_MULTI:
    pdf_path = os.path.join(fig_dir_m, fname)
    png_path = pdf_path.replace('.pdf', '.png')
    if os.path.exists(pdf_path):
        os.system(f'convert -density 150 "{pdf_path}" "{png_path}" 2>/dev/null || '
                  f'pdftoppm -r 150 -png "{pdf_path}" "{png_path[:-4]}" 2>/dev/null')
        img_file = png_path if os.path.exists(png_path) else (
            png_path[:-4] + '-1.png' if os.path.exists(png_path[:-4] + '-1.png') else None)
        if img_file and os.path.exists(img_file):
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.imshow(mpimg.imread(img_file))
            ax.axis('off'); ax.set_title(f'[multi] {title}', fontsize=10)
            plt.tight_layout(); plt.show()
        else:
            print(f'[multi] {title}: PDF saved at {pdf_path}')
    else:
        print(f'[multi] {title}: not found')

loss_png = glob.glob(os.path.join(fig_dir_m, 'loss_curves_*.png'))
if loss_png:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.imshow(mpimg.imread(loss_png[0]))
    ax.axis('off'); ax.set_title('[multi] Training loss curves', fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Cell M3: Multi-class numeric summary ──────────────────────────────────
import json
import pandas as pd

with open(os.path.join(run_dir_multi, 'metrics.json')) as f:
    mm = json.load(f)

n_list_m   = mm['n_list']
rho_list_m = mm['rho_list']
dets_m = list(mm['vs_n'].keys())

print('=== vs-n  AUC (multi-class) ===')
df_auc_mn = pd.DataFrame({d: mm['vs_n'][d]['auc'] for d in dets_m}, index=n_list_m)
df_auc_mn.index.name = 'n'
print(df_auc_mn.round(3).to_string())

print('\n=== vs-ρ  AUC (multi-class) ===')
df_auc_mr = pd.DataFrame({d: mm['vs_rho'][d]['auc'] for d in dets_m}, index=rho_list_m)
df_auc_mr.index.name = 'rho'
print(df_auc_mr.round(3).to_string())

if 'roc_at_nmax' in mm:
    n_roc_m = n_list_m[-1]
    print(f'\n=== ROC AUC at n={n_roc_m} (multi-class) ===')
    for d, v in mm['roc_at_nmax'].items():
        print(f'  {d:12s}: AUC = {v["auc"]:.4f}')

---
## Single vs Multi — Side-by-side comparison

In [ ]:
# ── Cell C1: AUC vs n — single vs multi ───────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

COLORS = {'AMF':'#1f77b4','Reg-AMF':'#6baed6','GMM-GLRT':'#9467bd',
          'DSM':'#d62728','LRao':'#2ca02c'}
MARKERS = {'AMF':'o','Reg-AMF':'s','GMM-GLRT':'^','DSM':'D','LRao':'D'}
LW      = {'AMF':1.4,'Reg-AMF':1.4,'GMM-GLRT':1.4,'DSM':2.2,'LRao':2.2}

def _plot_auc_compare(ax, n_list, metrics_dict, dets, title):
    x = np.array(n_list, dtype=float)
    for d in dets:
        ys = metrics_dict['vs_n'][d]['auc']
        ax.plot(x, ys, color=COLORS.get(d,'#444'), lw=LW.get(d,1.4),
                marker=MARKERS.get(d,'o'), label=d)
    ax.set_xscale('log'); ax.set_xlabel('n'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_plot_auc_compare(ax1, metrics_single['n_list'], metrics_single,
                  list(metrics_single['vs_n'].keys()), 'AUC vs n  [single-class]')
_plot_auc_compare(ax2, metrics_multi['n_list'],  metrics_multi,
                  list(metrics_multi['vs_n'].keys()),  'AUC vs n  [multi-class]')
fig.suptitle('AUC vs training size — single vs multi background', fontsize=10)
fig.tight_layout()
plt.savefig(os.path.join(RESULTS, 'compare_auc_vs_n.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell C2: AUC vs ρ — single vs multi ───────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

def _plot_auc_rho_compare(ax, rho_list, metrics_dict, dets, title):
    x = np.array(rho_list, dtype=float)
    for d in dets:
        ys = metrics_dict['vs_rho'][d]['auc']
        ax.plot(x, ys, color=COLORS.get(d,'#444'), lw=LW.get(d,1.4),
                marker=MARKERS.get(d,'o'), label=d)
    ax.set_xscale('log'); ax.set_xlabel(r'$\rho$'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_plot_auc_rho_compare(ax1, metrics_single['rho_list'], metrics_single,
                      list(metrics_single['vs_rho'].keys()), f'AUC vs ρ  [single, n={metrics_single["n_fixed"]}]')
_plot_auc_rho_compare(ax2, metrics_multi['rho_list'],  metrics_multi,
                      list(metrics_multi['vs_rho'].keys()),  f'AUC vs ρ  [multi, n={metrics_multi["n_fixed"]}]')
fig.suptitle(r'AUC vs DSM noise level $\rho$ — single vs multi background', fontsize=10)
fig.tight_layout()
plt.savefig(os.path.join(RESULTS, 'compare_auc_vs_rho.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell D1: Download all figures as a zip ────────────────────────────────
import zipfile, os
from google.colab import files

ZIP_PATH = '/content/iid_figures.zip'

to_zip = []
for run_dir in [run_dir_single, run_dir_multi]:
    fig_dir = os.path.join(run_dir, 'figures')
    for ext in ['*.pdf', '*.png']:
        to_zip.extend(__import__('glob').glob(os.path.join(fig_dir, ext)))
# Also include the compare figures
for cmp in ['compare_auc_vs_n.pdf', 'compare_auc_vs_rho.pdf']:
    p = os.path.join(RESULTS, cmp)
    if os.path.exists(p): to_zip.append(p)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in to_zip:
        # Store with a short relative name: single/... or multi/...
        rel = os.path.relpath(p, os.path.dirname(run_dir_single.rstrip('/')))
        z.write(p, rel)

print(f'Zipped {len(to_zip)} files → {ZIP_PATH}')
files.download(ZIP_PATH)